## Date Handling of Input Files

> The data contains periods in CET/CEST format.
	

* we need to have start and end date in one consitent time zone
* we need to split the period into start and end date of the period 

"01/01/2015 00:00:00 - 01/01/2015 00:15:00"


## Setup

All the manipulations and plots in this notebook can be created with standard libraries such as matplotlib, statsmodels etc. 

In [11]:
# Main data packages. 
import numpy as np
import pandas as pd


## Import Data 

The data for this notebook was downloaded from the [meteoblue website](https://www.meteoblue.com/en/weather/archive/export/basel_switzerland_2661604) and consits of weather data for the city of Basel from 2008 till 2020. 

In [46]:
files = [
    "../../data/fuel_prices/Dutch_TTF_Natural_Gas_Futures_Historische_Daten_Daily_2018-2025.csv",
]

df = pd.concat((pd.read_csv(f, delimiter=",") for f in files), ignore_index=True)


In [39]:
df.shape  #(541275, 6)

(2016, 7)

In [40]:
df.columns

Index(['Datum', 'Zuletzt', 'Eröffn.', 'Hoch', 'Tief', 'Vol.', '+/- %'], dtype='object')

In [41]:
df.head(4)

,Datum,Zuletzt,Eröffn.,Hoch,Tief,Vol.,+/- %
0,31.12.2025,"28,161","27,755","28,430","27,755","0,03K","1,42%"
1,30.12.2025,"27,766","27,766","27,766","27,766",NaN,"-2,69%"
2,29.12.2025,"28,534","28,450","29,000","28,420","0,01K","1,56%"
3,26.12.2025,"28,095","28,095","28,095","28,095",NaN,"0,00%"


In [47]:
# Create working copy of dataframe
df = df[["Datum","Eröffn."]].rename(  #"Intraday (MW)","Current (MW)",  , "Zuletzt"
    columns={
        'Datum':'date',
        'Eröffn.': 'price_gas',  
        "#Zuletzt": "price_gas_close",
    }
)
df.head()

,date,price_gas
0,31.12.2025,"27,755"
1,30.12.2025,"27,766"
2,29.12.2025,"28,450"
3,26.12.2025,"28,095"
4,24.12.2025,"28,220"


In [48]:
df["date"] = pd.to_datetime(df["date"], errors="coerce", dayfirst=True)
#df["price_gas"] = pd.to_numeric(df["price_gas"], errors="coerce")
df["price_gas"] = pd.to_numeric(
    df["price_gas"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)


In [49]:
df

,date,price_gas
0,2025-12-31,27.755
1,2025-12-30,27.766
2,2025-12-29,28.450
3,2025-12-26,28.095
4,2025-12-24,28.220
...,...,...
2011,2018-01-08,19.050
2012,2018-01-05,18.915
2013,2018-01-04,19.200
2014,2018-01-03,19.325


In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2016 entries, 0 to 2015
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       2016 non-null   datetime64[ns]
 1   price_gas  2016 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 31.6 KB


In [51]:
# Check NaN only in one column ('price')
rows_with_missing_price = df[df["price_gas"].isna()]
rows_with_missing_price.tail(100)

,date,price_gas


## saving our files by hour

In [52]:
df.to_csv("../../data_cleaned/by_source/07_prices_gas.csv", index=False)